# Verifying a reactive module with `uv run verith`

`verith` is the command-line front end of the Lean pipeline. It takes a
**module file** — a Python file exposing a callable that returns a
`zrth.Module` — and writes a Lean 4 project holding the module's transition
system *and* a machine-checkable **certificate** for one temporal property:

```
module.py ──verith──▶ <OUT>/Rea/              a Lean 4 package
                        ├── System/           the module: init / update, in five encodings
                        ├── System/Data.lean  the certificate data: init_pre, update_pre, inv, P, ranking
                        └── Certificate/      the proof obligations, with generated tactic scripts
                                    │
                                    ▼   lake build Certificate
                              proved, or not
```

A certificate proves exactly one property under exactly one proof rule, and
the flag you pass the formula under **is** the choice of rule:

| flag | property | rule | the certificate consists of |
|---|---|---|---|
| `--safety P` | `G P` — `P` holds in every reachable state | `rule_globally` | an inductive **invariant** that implies `P` |
| `--buchi P` | `G (F P)` — `P` holds infinitely often | `rule_buchi` | an inductive **invariant** *and* a **ranking function** that strictly decreases wherever `¬P` |

Properties, invariants, ranking functions and preconditions are all written as
**SMT-LIB 2** expressions over the module's wires:

* `s0, s1, …` — the controlled (state) wires, in the order the module declares them
* `e0, e1, …` — the external inputs at the next step; `el0, el1, …` — the latched ones
* components of a vector wire take a tuple selector: `((_ tuple.select 1) e0)`

**What this notebook does**

1. **Setup** — paths, keys, and the one shared output directory
2. **Safety** on a module written in the Python API
3. **Büchi** (liveness) on a module written in the Python API
4. A module read out of a **gymnasium** environment
5. **When verification fails** — an invalid invariant, and a missing precondition
6. Letting an **LLM** find the certificate (`--infer`)
7. Letting **ic3ia** find the invariant (`--fbk-proveit`)
8. A flag reference

Every module used below is a fixture that already lives in `python/tests/`, so
nothing has to be written first.

## 1. Setup

Everything configurable is in the next cell: where the checkout is, where the
generated projects go, and the paths and keys of the two optional routes
(`--infer`, `--fbk-proveit`).

**One output directory for every command.** `verith` regenerates
`<OUT>/<PROJ>` in place — it never wipes the directory — so the project's
`.lake` survives from one example to the next, and with it every `.olean` of
Mathlib, cslib, lean-smt, `Core` and `ZerothHammer`. Only `System/*` and
`Certificate/*` recompile per example: the first `lake build` below costs
about 15 s and each later one about 10 s, against an hour from cold.

Two things make that work:

* `<OUT>/<PROJ>/.lake` is a symlink to one shared build directory whose
  `packages` points at the already-built packages of `python/tests/lean`.
  Build those once with `cd python/tests/lean && lake build`.
* Keep the `.noindex` suffix on `OUT` if you are on macOS. A build writes tens
  of thousands of `.olean` files; Spotlight indexing them has turned a 9 s
  build into a 928 s one, and a directory whose name ends in `.noindex` is
  skipped.

One shared build directory takes **one writer** — don't run a second notebook
or `tests/limits/run_limits.py` against the same `OUT` at the same time. Module
names are identical in every generated project, so concurrent builds serve each
other's oleans, and the symptom is `unknown constant 'hrank'`, which reads
exactly like a codegen bug.

In [ ]:
# ═══════════════════════════ SETUP ═══════════════════════════════════════
import os, re, shlex, shutil, subprocess, time
from pathlib import Path

# The repository checkout (this notebook lives in <repo>/tutorials/).
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "python" / "zrth").is_dir())
PY   = REPO / "python"          # every `uv run verith` below runs from here

# ── one output directory, one project name, for every command ────────────
OUT     = Path(os.environ.get("VERITH_TUTORIAL_OUT", "/tmp/verith-tutorial.noindex"))
PROJ    = "Rea"                 # the Lean package name (-p)
PROJECT = OUT / PROJ            # what `-o OUT -p Rea` regenerates, every time

# ── the already-built Mathlib / cslib / lean-smt to borrow ───────────────
WARM     = PY / "tests" / "lean"              # cd there and `lake build`, once
PACKAGES = WARM / ".lake" / "packages"
MANIFEST = WARM / "lake-manifest.json"

# ── the --fbk-proveit route (optional): ic3ia finds the invariant ────────
# Building these three is documented in python/tests/lean/fbk/README.md.
PROVEIT = Path.home() / "zeroth" / "proof-prototyping" / "lean-ltl-certifying"
IC3IA   = Path.home() / "zeroth" / "fbk" / "ic3ia" / "build" / "ic3ia"
MATHSAT = Path.home() / "zeroth" / "fbk" / "mathsat" / "python"   # mathsat.py + _mathsat*.so
if MATHSAT.is_dir():
    os.environ["PYTHONPATH"] = str(MATHSAT)   # vmt2lean.py does `from mathsat import *`

# ── the --infer route (optional): an LLM proposes invariant and ranking ──
# Claude is the default backend. Any OpenAI-compatible endpoint (Ollama, vLLM,
# OpenRouter) works instead, through --base-url + --model; see section 6.
KEYFILE = REPO / "CLAUDE_KEY.txt"             # a key kept in the checkout, if you do that
if not os.environ.get("ANTHROPIC_API_KEY") and KEYFILE.exists():
    os.environ["ANTHROPIC_API_KEY"] = KEYFILE.read_text().strip()
# os.environ["OPENAI_API_KEY"]     = "sk-..."       # with --base-url
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."    # with --base-url https://openrouter.ai/api/v1

# ── wire up the shared .lake, once ───────────────────────────────────────
SHARED = OUT / "shared_lake"                  # one build dir for the whole notebook
SHARED.mkdir(parents=True, exist_ok=True)
if PACKAGES.is_dir() and not (SHARED / "packages").is_symlink():
    (SHARED / "packages").symlink_to(PACKAGES)

# ── what is available in this environment? ───────────────────────────────
def _importable(mod):
    import importlib.util
    return importlib.util.find_spec(mod) is not None

for ok, name, why in [
    (bool(shutil.which("lake")),                "lake (Lean 4)",       "needed to *check* a certificate"),
    (PACKAGES.is_dir(),                         "warm .lake packages", f"cd {WARM} && lake build"),
    (_importable("cvc5"),                       "cvc5",                "--pre-check, --smt-tactics, --infer ai-cegar"),
    (bool(os.environ.get("ANTHROPIC_API_KEY")), "ANTHROPIC_API_KEY",   "--infer with the default backend"),
    (_importable("anthropic"),                  "anthropic package",   "uv sync --extra ai"),
    (IC3IA.exists(),                            "ic3ia",               "--fbk-proveit"),
    (PROVEIT.is_dir(),                          "lean-ltl-certifying", "--fbk-proveit"),
    (MATHSAT.is_dir(),                          "mathsat bindings",    "--fbk-proveit"),
]:
    print(f"  {'yes' if ok else ' - '}  {name:22}  {'' if ok else why}")

print(f"\n  projects -> {PROJECT}")

  yes  lake (Lean 4)           
  yes  warm .lake packages     
  yes  cvc5                    
  yes  ANTHROPIC_API_KEY       
  yes  anthropic package       
  yes  ic3ia                   
  yes  lean-ltl-certifying     
  yes  mathsat bindings        

  projects -> /tmp/verith-tutorial.noindex/Rea


Now the helpers the rest of the notebook uses. There is nothing clever in
them: `verith()` is `uv run verith … -o OUT -p Rea`, and `lake_build()` is
`lake build Certificate` in that one project. Both print only the lines worth
reading — `verith` also dumps the whole module and every file it wrote.

`verith --build-cert` does the build itself in one flag, which is what you
would type at a terminal. It runs `lake update` first, and that re-queries
every dependency's git remote; `lake_build()` copies the warm manifest
instead, so the examples below never touch the network.

In [ ]:
ANSI = re.compile(r"\x1b\[[0-9;]*m")


def run(cmd, cwd, timeout=900):
    """Run a command; return (exit code, stdout + stderr).

    A timeout is reported like any other failure rather than raised: the one
    step here that can hang for reasons of its own is the LLM call behind
    `--infer`, and a stalled notebook is a worse outcome than a failed cell.
    """
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return 124, f"error: timed out after {timeout}s: {' '.join(cmd)}"
    return p.returncode, ANSI.sub("", p.stdout + p.stderr)


def grep(text, keep, limit=30):
    """Print the lines of `text` mentioning any of `keep`. Returns them."""
    hits = [l.rstrip() for l in text.splitlines() if any(k in l for k in keep)]
    for l in hits[:limit]:
        print(l)
    if len(hits) > limit:
        print(f"   ... and {len(hits) - limit} more line(s)")
    return hits


# What verith says about the certificate, as opposed to the module dump and
# the list of files it wrote.
SAYS = ("pre-check", "holds", "REFUTED", "refuted", "[CEGAR]", "[nuterm]",
        "inv:", "ranking:",
        "error:", "Project ready", "Certificate built", "property is",
        "invariant witness", "certificate:", "Installed", "Wrote the")


def verith(*args, keep=SAYS, limit=30, timeout=900):
    """`uv run verith <args> -o OUT -p Rea`, run from python/.

    Always the same -o/-p: each example regenerates the one project, so the
    Lean build reuses everything it compiled for the previous example.
    """
    # The routes write different files into the project -- --fbk-proveit adds
    # ProveIt/ and Certificate/Equivalence.lean -- and one route's leftovers
    # would end up in the next one's build.
    for stale in (PROJECT / "Certificate", PROJECT / "ProveIt"):
        shutil.rmtree(stale, ignore_errors=True)

    cmd = ["uv", "run", "verith", *map(str, args), "-o", str(OUT), "-p", PROJ]
    print("$ " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    t0 = time.perf_counter()
    rc, out = run(cmd, cwd=PY, timeout=timeout)
    grep(out, keep, limit) if keep else print(out)
    # Section 7 compares two routes on the same module, and the wall clock is
    # half of what it compares.
    print(f"\n[{time.perf_counter() - t0:.1f}s, exit {rc}]")
    return rc, out


def lake_build(target="Certificate", manifest=MANIFEST, limit=30):
    """`lake build <target>` in the shared project.

    `.lake` is a symlink to one build directory whose `packages` are the warm
    ones, and `manifest` is copied in so lake resolves them offline. Pass
    `manifest=None` to run `lake update` first instead -- what `--build-cert`
    does, and what the --fbk-proveit route needs, its lakefile requiring a
    package the warm manifest has never seen.
    """
    lake = PROJECT / ".lake"
    if not lake.is_symlink():
        shutil.rmtree(lake, ignore_errors=True)
        lake.symlink_to(SHARED)
    if manifest is None:
        rc, out = run(["lake", "update"], cwd=PROJECT, timeout=1800)
        if rc != 0:
            print("lake update failed:")
            grep(out, ("error:",), 10)
            return rc, out
    else:
        shutil.copy2(manifest, PROJECT / "lake-manifest.json")

    print(f"$ lake build {target}      # in {PROJECT}\n")
    rc, out = run(["lake", "build", target], cwd=PROJECT, timeout=1800)
    grep(out, ("error:", "✖ ", "Build completed"), limit)
    # A build that succeeds while an obligation is still `sorry` is not a
    # proof: the tactic cascade leaves one where it cannot close a goal, and
    # lake reports that as a warning and exits 0. (lean-smt ships a `sorry` of
    # its own, which is not ours.)
    ours = [l.strip() for l in out.splitlines() if "sorry" in l and "Certificate/" in l]
    print(*ours, sep="\n")
    print("\n=>", "PROVED" if rc == 0 and not ours else "NOT PROVED", f"(lake exit {rc})")
    return rc, out


def show(path, keep=None, head=None, limit=40):
    """Print a file of the generated project, or the part of it that matters."""
    p = PROJECT / path
    text = ANSI.sub("", p.read_text())
    print(f"── {p} " + "─" * max(3, 62 - len(str(p))))
    if keep:
        grep(text, keep, limit)
    else:
        print("\n".join(text.splitlines()[:head]) if head else text)


def source(relpath, start=None, end=None):
    """Print a source file from python/ (or the slice between two markers)."""
    text = (PY / relpath).read_text()
    if start:
        text = text[text.index(start): text.index(end) if end else None].rstrip() + "\n"
    print(f"── {relpath} " + "─" * max(3, 62 - len(relpath)))
    print(text)

## 2. Safety: a module written in the Python API

`tests/limits/mods/m_countdown.py` is the smallest interesting module in the
repository: one integer state wire that counts down from 100 and wraps back to
100 at zero. It uses the **analyzer** front end — `convert_method` reads the
bytecode of two plain Python functions and turns them into the module's `init`
and `update` term lists.

In [ ]:
source("tests/limits/mods/m_countdown.py")

── tests/limits/mods/m_countdown.py ──────────────────────────────
"""LIA 1x1: x = 100, then x-1 each step, reset to 100 at 0. Baseline control."""
from zrth import Module, Int, LIA, Var, X
from zrth.analyzer import convert_method


def init():
    return 100


def update(old_x):
    if old_x == 0:
        return 100
    return old_x - 1


def module() -> Module:
    s = Var(Int([1, 1]))
    return Module.sequential(
        [s],
        convert_method(init, {}, [X(s)], theory=LIA),
        convert_method(update, {"old_x": s}, [X(s)], theory=LIA),
    )



The safety property is that the counter never leaves `[0, 100]`:

```
--safety '(and (>= s0 0) (<= s0 100))'
```

`s0` is the module's one controlled wire. Under `rule_globally` the whole
certificate is an **invariant** that holds initially, is preserved by `update`,
and is strong enough to imply the property — here the property is itself
inductive, so the same formula serves as both.

`--pre-check cvc5` asks cvc5 whether the obligations are actually true
*before* spending a Lean build on them. It answers in milliseconds and a
refutation comes with a counterexample; a failing `lake build` takes tens of
seconds and still cannot tell a wrong certificate from tactics that are merely
too weak.

In [ ]:
rc, _ = verith(
    "tests/limits/mods/m_countdown.py",
    "--safety",    "(and (>= s0 0) (<= s0 100))",
    "--invariant", "(and (>= s0 0) (<= s0 100))",
    "--pre-check", "cvc5",
)

$ uv run verith tests/limits/mods/m_countdown.py --safety '(and (>= s0 0) (<= s0 100))' --invariant '(and (>= s0 0) (<= s0 100))' --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         1 ms
   inv_imp_P holds         1 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.9s, exit 0]


Three obligations, all discharged by cvc5:

* `init_inv` — every initial state satisfies the invariant
* `step_inv` — `update` preserves it, i.e. it is inductive
* `inv_imp_P` — the invariant implies the property

The SMT-LIB predicates are compiled into Lean in `System/Data.lean`. `s0`
becomes `s 0 0`, the single entry of a `Mat Int 1 1`; a module with several
wires gets a product, and `s0, s1, s2, …` become `s.1, s.2.1, s.2.2.1, …`.

In [ ]:
show("System/Data.lean")

── /tmp/verith-tutorial.noindex/Rea/System/Data.lean ─────────────
import Core.Basic

def init_pre (e : (Unit) × (Unit)) : Prop := True

def update_pre (e : (Unit) × (Unit)) : Prop := True

def inv : (Mat Int 1 1) → Prop := fun s => (((s 0 0) ≥ 0) ∧ ((s 0 0) ≤ 100))

def P : (Mat Int 1 1) → Prop := fun s => (((s 0 0) ≥ 0) ∧ ((s 0 0) ≤ 100))

instance : DecidablePred P := fun s => by unfold P; first | infer_instance | dsimp; infer_instance




`Certificate/Certificate.lean` states those obligations against the Lean module
and attacks each with a tactic script generated from the shape of the module
and of the predicates — the header records what it keyed on.

In [ ]:
show("Certificate/Certificate.lean",
     keep=("-- state:", "-- predicates:", "def RM", "theorem ", "def lts", "def safety"))

── /tmp/verith-tutorial.noindex/Rea/Certificate/Certificate.lean ───
-- state: Bool+Int, 1 slot(s), 9 term(s)
-- predicates: branching, conjunctive, linear
def RM : ReactiveModule ((Unit) × (Unit)) ((Mat Int 1 1)) := {
theorem init_inv : ∀ s, RM.init_pre s → inv (RM.init s) := by
theorem step_inv : ∀ s e, (RM.update_pre e ∧ inv s) → inv (RM.update s e) := by
def lts := RM.toTS
theorem hinv' : lts.StateSet_isInductiveInitial inv := by
theorem hinv : lts.StateSet_isInvariant inv := by
theorem inv_imp_P : ∀ s, inv s → P s := by
theorem hP : lts.StateSet_isInvariant P := by
def safety := rule_globally


Now check it with Lean. This first build is the expensive one: it compiles
`Core`, `LeanAI` and `ZerothHammer` into the shared build directory, and every
later example in this notebook reuses them.

In [ ]:
rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


## 3. Büchi: the same idea, plus a ranking function

`tests/fixtures/counter.py` counts *up*: 0, 1, …, 9, then back to 0. "The
counter is at zero infinitely often" is a liveness property, `G (F (s0 = 0))`,
and it goes under `--buchi`.

An invariant alone cannot prove it — an invariant says where the system *may*
be, never that it gets anywhere. `rule_buchi` therefore also takes a **ranking
function**: a natural number, non-negative under the invariant, that strictly
decreases on every step taken from a state where the property is false. Here
`10 - s0` counts the steps left until the wrap.

In [ ]:
source("tests/fixtures/counter.py")

── tests/fixtures/counter.py ─────────────────────────────────────
"""Simple counter reactive module fixture for CLI tests.

Python semantics:
    x starts at 0, increments each step, resets to 0 when it reaches 10.
    Property: x == 0 holds infinitely often.
"""
from zrth import Module, Int, LIA, Var, X
from zrth.analyzer import convert_method


def init():
    return 0


def update(old_x):
    x = old_x + 1
    if x == 10:
        return 0
    return x


def module() -> Module:
    state = Var(Int([1, 1]))
    init_terms = convert_method(init, {}, [X(state)], theory=LIA)
    update_terms = convert_method(update, {"old_x": state}, [X(state)], theory=LIA)
    return Module.sequential([state], init_terms, update_terms)



In [ ]:
rc, _ = verith(
    "tests/fixtures/counter.py",
    "--buchi",     "(= s0 0)",
    "--invariant", "(and (>= s0 0) (<= s0 9))",
    "--ranking",   "(ite (= s0 0) 0 (- 10 s0))",
    "--pre-check", "cvc5",
)

$ uv run verith tests/fixtures/counter.py --buchi '(= s0 0)' --invariant '(and (>= s0 0) (<= s0 9))' --ranking '(ite (= s0 0) 0 (- 10 s0))' --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         1 ms
   hrank     holds         2 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[1.0s, exit 0]


The obligations are now `init_inv`, `step_inv` and **`hrank`**, the decrease.
`inv_imp_P` is gone: a Büchi property is not implied by the invariant, it is
*reached*.

`Data.lean` gains a `ranking` definition, and the certificate ends in
`rule_buchi` rather than `rule_globally`:

In [ ]:
show("System/Data.lean", keep=("def ", "instance"))
print()
show("Certificate/Certificate.lean", keep=("theorem ", "def lts", "def buchi"))

── /tmp/verith-tutorial.noindex/Rea/System/Data.lean ─────────────
def init_pre (e : (Unit) × (Unit)) : Prop := True
def update_pre (e : (Unit) × (Unit)) : Prop := True
def inv : (Mat Int 1 1) → Prop := fun s => (((s 0 0) ≥ 0) ∧ ((s 0 0) ≤ 9))
def P : (Mat Int 1 1) → Prop := fun s => ((s 0 0) = 0)
instance : DecidablePred P := fun s => by unfold P; first | infer_instance | dsimp; infer_instance
def ranking : (Mat Int 1 1) → Nat := fun s => (((if ((s 0 0) = 0) then 0 else (10 - (s 0 0))) : Int)).toNat

── /tmp/verith-tutorial.noindex/Rea/Certificate/Certificate.lean ───
theorem init_inv : ∀ s, RM.init_pre s → inv (RM.init s) := by
theorem step_inv : ∀ s e, (RM.update_pre e ∧ inv s) → inv (RM.update s e) := by
def lts := RM.toTS
theorem hinv' : lts.StateSet_isInductiveInitial inv := by
theorem hinv : lts.StateSet_isInvariant inv := by
theorem hrank : ∀ s s', (inv s ∧ ¬(P s) ∧ (∃ l, lts.Tr s l s')) →
def buchi := rule_buchi


In [ ]:
rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


## 4. A module read out of a gymnasium environment

Nothing above is specific to hand-written modules. `zrth.gym.Env` extracts a
module from a plain `gymnasium.Env` by symbolically analyzing its `reset` and
`step` methods, and the result is a module file like any other.

`tests/fixtures/simple_env.py` wraps `SimpleEnv` — a three-cell chain where the
agent walks left or right and the goal is cell 2.

In [ ]:
source("tests/fixtures/simple_env.py")
source("tests/gym/environments.py", start="class SimpleEnv", end="class TwoBitCounterEnv")

── tests/fixtures/simple_env.py ──────────────────────────────────
"""SimpleEnv gym wrapper fixture.

Chain environment: state ∈ {0,1,2}, action moves left/right.
Property: state reaches 2 infinitely often.
"""
from zrth.gym import Env
from zrth import Module, Real
from tests.gym.environments import SimpleEnv


def module() -> Module:
    # `state` is SimpleEnv's only private attribute; its sort must be explicit.
    return Env(SimpleEnv(), attrs=Real([1, 1]))

── tests/gym/environments.py ─────────────────────────────────────
class SimpleEnv(gym.Env):
    """Chain env: state moves left/right; reward 1 at goal (state == 2)."""

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Discrete(2)

    def _get_observation(self):
        return 1 if self.state == 2 else 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0
        return self._get_observatio

`reset` becomes the module's `init` and `step` its `update`. Instance
attributes written by either (`self.state`) become **private** wires; the
observation, reward, `terminated` and `truncated` returns become **interface**
wires; the action becomes an **external** input.

### Which wire is `s0`?

`s0 … sN-1` are the module's controlled wires in order, and `verith` prints
that order — the `controls` line of the module dump, also written to
`dbg/system.txt`. Generate the project bare, with no property, and read it:

In [ ]:
rc, _ = verith("tests/fixtures/simple_env.py")
print()
show("dbg/system.txt", head=11)

$ uv run verith tests/fixtures/simple_env.py -o /tmp/verith-tutorial.noindex -p Rea



Project ready at: /tmp/verith-tutorial.noindex/Rea

[1.0s, exit 0]

── /tmp/verith-tutorial.noindex/Rea/dbg/system.txt ───────────────
module
  external
    #3 : Real([1,2])
  interface
    #6 : Real([1,1])
    #9 : Real([1,1])
    #12 : Bool([1,1])
    #15 : Bool([1,1])
  private
    #0 : Real([1,1])
  atom controls #0, #6, #9, #12, #15 reads #0, #3


`atom controls #0, #6, #9, #12, #15` is the state order, so for this module:

| SMT var | wire | Lean type | in `Data.lean` |
|---|---|---|---|
| `s0` | `self.state` (private) | `Mat Real 1 1` | `s.1` |
| `s1` | observation | `Mat Real 1 1` | `s.2.1` |
| `s2` | reward | `Mat Real 1 1` | `s.2.2.1` |
| `s3` | terminated | `Mat Bool 1 1` | `s.2.2.2.1` |
| `s4` | truncated | `Mat Bool 1 1` | `s.2.2.2.2` |
| `e0` / `el0` | action, one-hot of width 2 | `Mat Real 1 2` | — |

The state tuple in `System/System.lean`'s `update` signature matches it
component by component, and once a property is compiled in you can confirm the
mapping by reading the predicate in `System/Data.lean`.

Note that the arithmetic is **real**, not integer: under the default theory a
`Discrete` action space becomes a one-hot real vector (read back with `argmax`)
and rewards are real-valued. Hence `0.0` and `2.0` in the property below rather
than `0` and `2`.

### A safety certificate for the chain

"The chain position never leaves `[0, 2]`" — true whatever the agent does,
since `step` clamps with `min` and `max`:

In [ ]:
rc, _ = verith(
    "tests/fixtures/simple_env.py",
    "--safety",    "(and (>= s0 0.0) (<= s0 2.0))",
    "--invariant", "(and (>= s0 0.0) (<= s0 2.0))",
    "--pre-check", "cvc5",
)
print()
show("System/Data.lean", keep=("def inv", "def P"))

$ uv run verith tests/fixtures/simple_env.py --safety '(and (>= s0 0.0) (<= s0 2.0))' --invariant '(and (>= s0 0.0) (<= s0 2.0))' --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         2 ms
   step_inv  holds         2 ms
   inv_imp_P holds         1 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[1.0s, exit 0]

── /tmp/verith-tutorial.noindex/Rea/System/Data.lean ─────────────
def inv : (Mat Real 1 1) × (Mat Real 1 1) × (Mat Real 1 1) × (Mat Bool 1 1) × (Mat Bool 1 1) → Prop := fun s => (((s.1 0 0) ≥ (0 : Real)) ∧ ((s.1 0 0) ≤ (2 : Real)))
def P : (Mat Real 1 1) × (Mat Real 1 1) × (Mat Real 1 1) × (Mat Bool 1 1) × (Mat Bool 1 1) → Prop := fun s => (((s.1 0 0) ≥ (0 : Real)) ∧ ((s.1 0 0) ≤ (2 : Real)))


In [ ]:
rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


Two things worth knowing about the real-valued case before reaching for a Büchi
property here:

* Every Lean definition becomes `noncomputable`, and the proofs lean on
  `linarith` rather than the more robust `omega`.
* An *interval* invariant is not inductive enough for a ranking argument over
  the reals. `(and (>= s0 0.0) (<= s0 2.0))` admits `s0 = 5/4`, and cvc5 duly
  refutes `hrank` there. Pinning the reachable values instead —
  `(or (= s0 0.0) (= s0 1.0) (= s0 2.0))` — makes cvc5 accept all three
  obligations, but the generated tactic cascade still does not close `hrank` on
  this module: `argmax` over real wires is a known gap.

If the dynamics are genuinely discrete, design the environment so the
extraction stays in integer arithmetic: pass `theory=LIA` to `Env`, use an
integer `Box` action space rather than `Discrete`, and return integer rewards.
[`verith_gym.md`](verith_gym.md) works that variant through in full.

## 5. When verification fails

Two mistakes account for most of what goes wrong, and each looks quite
different at the two places you can catch it.

### 5a. An invalid invariant

Take the counter of section 3 and claim it stays below 5. The property is still
true and the ranking is still fine — but the invariant is **not inductive**:
from 5 the counter steps to 6, which is outside it.

In [ ]:
rc, _ = verith(
    "tests/fixtures/counter.py",
    "--buchi",     "(= s0 0)",
    "--invariant", "(and (>= s0 0) (<= s0 5))",     # wrong: the counter runs to 9
    "--ranking",   "(ite (= s0 0) 0 (- 10 s0))",
    "--pre-check", "cvc5",
)

$ uv run verith tests/fixtures/counter.py --buchi '(= s0 0)' --invariant '(and (>= s0 0) (<= s0 5))' --ranking '(ite (= s0 0) 0 (- 10 s0))' --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  REFUTED       1 ms   counterexample: s0 = 5
   hrank     holds         2 ms
   1 obligation(s) refuted (step_inv): the certificate is wrong, not merely hard -- Lean cannot close it
Project ready at: /tmp/verith-tutorial.noindex/Rea

[1.0s, exit 0]


`step_inv REFUTED — counterexample: s0 = 5`, and that is the whole diagnosis:
at `s0 = 5` the invariant holds and after one step it does not. `verith` still
generates the project — a refuted pre-check is a warning, not an abort — so we
can look at what Lean makes of the same mistake. `lake_build` returns the raw
output as well, so we can print the goal that survived:

In [ ]:
rc, out = lake_build()

lines = out.splitlines()
first = next((i for i, l in enumerate(lines) if "linarith failed" in l), None)
if first is not None:
    print("\nthe goal linarith could not close:\n")
    print("\n".join(lines[first + 1: first + 9]))

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



✖ [3464/3466] Building Certificate.Certificate (3.4s)
error: Certificate/Certificate.lean:75:28: linarith failed to find a contradiction
error: Lean exited with code 1
error: build failed


=> NOT PROVED (lake exit 1)

the goal linarith could not close:

case neg.inl
s : Mat ℤ 1 1
e : Unit × Unit
left✝ : 0 ≤ s 0 0
right✝ : s 0 0 ≤ 5
h✝ : s 0 0 + 1 < 10
⊢ False
failed


`linarith failed to find a contradiction`, and underneath it the goal: from
`0 ≤ s 0 0`, `s 0 0 ≤ 5` and `s 0 0 + 1 < 10`, derive `False`. Accurate as a
report, useless as a diagnosis — it does not say whether the certificate is
wrong or the tactic is too weak, and it took a Lean build to say it. **Run
`--pre-check cvc5` first.**

### 5b. A missing precondition

`tests/limits/mods/m_relu_input.py` is a counter driven by an external input:
`x' = relu(x - e)`, starting at 5. It reaches 0 and stays there — but only if
the environment feeds it `e >= 1`. Nothing constrains an external input, so
without saying so the certificate is simply false.

In [ ]:
source("tests/limits/mods/m_relu_input.py")

── tests/limits/mods/m_relu_input.py ─────────────────────────────
"""ReLU plus an external input: x' = relu(x - e), x0 = 5, e supplied each step.

Sound only under a precondition `e >= 1`; without it the ranking does not
decrease. Used twice, with and without `--pre`.
"""
import torch
from zrth import Module, Term, Wire, Int, LIA, Var, X


def module() -> Module:
    x = Var(Int([1, 1]))
    e = Var(Int([1, 1]))
    diff = Wire(Int([1, 1]))
    init = [Term(LIA.Int(torch.tensor([[5]])), [X(x)])]
    update = [
        Term(LIA.Sub(), [diff], [x, X(e)]),
        Term(LIA.ReLU(), [X(x)], [diff]),
    ]
    return Module.sequential([x, e], init, update)



In [ ]:
rc, _ = verith(
    "tests/limits/mods/m_relu_input.py",
    "--buchi",     "(= s0 0)",
    "--invariant", "(and (>= s0 0) (<= s0 5))",
    "--ranking",   "s0",
    "--pre-check", "cvc5",
)

$ uv run verith tests/limits/mods/m_relu_input.py --buchi '(= s0 0)' --invariant '(and (>= s0 0) (<= s0 5))' --ranking s0 --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  REFUTED       1 ms   counterexample: s0 = 0
   hrank     REFUTED       1 ms   counterexample: s0 = 1
   2 obligation(s) refuted (step_inv, hrank): the certificate is wrong, not merely hard -- Lean cannot close it
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.9s, exit 0]


Two refutations this time, both the same missing hypothesis:

* `step_inv` at `s0 = 0` — with `e` negative, `relu(0 - e)` climbs above 5 and
  the bound breaks;
* `hrank` at `s0 = 1` — with `e = 0` the state does not move, so the ranking
  does not decrease.

Lean reports it as two failed `linarith` calls, one per obligation:

In [ ]:
rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



✖ [3464/3466] Building Certificate.Certificate (3.3s)
error: Certificate/Certificate.lean:75:28: linarith failed to find a contradiction
error: Certificate/Certificate.lean:109:29: linarith failed to find a contradiction
error: Lean exited with code 1
error: build failed


=> NOT PROVED (lake exit 1)


`--pre` supplies the missing hypothesis as a precondition over the input wires.
It is added to `init_pre` and `update_pre` alike, and every obligation may then
assume it:

In [ ]:
rc, _ = verith(
    "tests/limits/mods/m_relu_input.py",
    "--buchi",     "(= s0 0)",
    "--invariant", "(and (>= s0 0) (<= s0 5))",
    "--ranking",   "s0",
    "--pre",       "(>= e0 1)",          # the environment always decrements
    "--pre-check", "cvc5",
)
print()
show("System/Data.lean", keep=("def init_pre", "def update_pre"))

$ uv run verith tests/limits/mods/m_relu_input.py --buchi '(= s0 0)' --invariant '(and (>= s0 0) (<= s0 5))' --ranking s0 --pre '(>= e0 1)' --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         1 ms
   hrank     holds         1 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.9s, exit 0]

── /tmp/verith-tutorial.noindex/Rea/System/Data.lean ─────────────
def init_pre : ((Mat Int 1 1)) × ((Mat Int 1 1)) → Prop := fun e => ((e.2 0 0) ≥ 1)
def update_pre : ((Mat Int 1 1)) × ((Mat Int 1 1)) → Prop := fun e => ((e.2 0 0) ≥ 1)


In [ ]:
rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


**`e0` or `el0`?** `e0` is the input at the next step, `el0` the one latched at
the previous step, and the precondition has to speak about whichever copy
`update` actually reads. This module reads `X(e)`, the next value, so `e0` is
right. A module extracted from a gym environment reads the *latched* action, so
there the precondition goes over `el0` — the `update` block of the module dump
shows which wire each term reads.

One more thing that catches people: the obligations are quantified over **every
state satisfying the invariant**, not over the reachable ones. A ranking that
decreases along all reachable runs but not from some unreachable state the
invariant admits is a wrong certificate, and that is the usual reason a
correct-looking case fails.

## 6. Letting an LLM find the certificate (`--infer`)

Writing the invariant and the ranking function by hand is the hard part, and it
is the part `verith` can search for. With `--infer` you supply only the property
(and any precondition) and an LLM proposes the rest.

The default mode, `ai-cegar`, checks every proposal with cvc5 and feeds the
counterexamples back to the model until the certificate holds — so a wrong
guess costs one SMT query, not a Lean build. (`--infer ai` skips cvc5 and relies
on the model checking itself; it cannot serve `--safety`.)

Needs `ANTHROPIC_API_KEY` and `uv sync --extra ai` for the default backend.
Section 7 is the other `--infer`, which needs neither.

In [ ]:
INFERRED = False
if os.environ.get("ANTHROPIC_API_KEY"):
    rc, _ = verith(
        "tests/limits/mods/m_countdown.py",
        "--buchi", "(= s0 0)",
        "--infer",                       # no --invariant, no --ranking
        timeout=300,                     # the one step that can stall on a remote
    )
    INFERRED = rc == 0
    if INFERRED:
        print()
        show("System/Data.lean", keep=("def inv", "def ranking"))
else:
    print("no ANTHROPIC_API_KEY -- skipping (see the setup cell)")

$ uv run verith tests/limits/mods/m_countdown.py --buchi '(= s0 0)' --infer -o /tmp/verith-tutorial.noindex -p Rea



[CEGAR] attempt 0
  inv: (and (<= 0 s0) (<= s0 100))
  ranking: s0
[CEGAR] all obligations UNSAT — accepted
Project ready at: /tmp/verith-tutorial.noindex/Rea

[5.7s, exit 0]

── /tmp/verith-tutorial.noindex/Rea/System/Data.lean ─────────────
def inv : (Mat Int 1 1) → Prop := fun s => ((0 ≤ (s 0 0)) ∧ ((s 0 0) ≤ 100))
def ranking : (Mat Int 1 1) → Nat := fun s => (((s 0 0) : Int)).toNat


In [ ]:
if INFERRED:
    rc, _ = lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


The inferred predicates are printed as SMT-LIB, so a good result can be frozen
by passing it back with `--invariant` / `--ranking` on later runs. The two modes
mix: fixing the invariant by hand makes `--infer` search only for a ranking
function, and fixing **both** makes no LLM call at all — the CEGAR loop just
verifies your certificate with cvc5, which is `--pre-check` by another route.

For a local model or another provider, add `--base-url` (and `--model`); the
only extra dependency is the `openai` client, `uv sync --extra ai-local`:

```bash
# Ollama
uv run verith tests/limits/mods/m_countdown.py --buchi '(= s0 0)' --infer \
    --model qwen3-coder --base-url http://localhost:11434/v1 -o OUT -p Rea

# OpenRouter (key from OPENROUTER_API_KEY or OPENAI_API_KEY)
uv run verith tests/limits/mods/m_countdown.py --buchi '(= s0 0)' --infer \
    --model anthropic/claude-haiku-4.5 --base-url https://openrouter.ai/api/v1 -o OUT -p Rea
```

A smaller model proposes worse invariants, which is exactly what the cvc5 loop
absorbs — it just takes more rounds.

## 7. Learning the certificate instead (`--infer nuterm`)

`--infer nuterm` answers the same question with no LLM in it, and the two
halves of the certificate come from two different places.

The **invariant** is Houdini's: seed a lattice of candidate facts about the
state — signs, pairwise relations, and the constants the program itself
mentions — drop the ones that do not hold at entry or are not preserved by a
step, and certify what survives. The **ranking function** is *learned*: a small
ReLU network is trained on rollouts of the rounds where the property fails, its
weights are rounded to integers, and the candidate is composed back into the
module as two ordinary atoms, `V(s)` and `V(s')`. That composed module goes to a
Farkas/CEGAR decision procedure over its own wires, which certifies the drop or
rejects the candidate and sends the trainer round again.

Only a candidate the procedure certifies comes back. So unlike the LLM routes,
what `verith` is handed here has *already* been proved, and `--pre-check` below
is an independent confirmation rather than the first check of it.

It needs no key, no network and no ic3ia build, and it is deterministic: the
same module and the same seed learn the same rank. What it does need is a state
of scalar integers and a transition that reads no nondeterministic input —
anything else it refuses by name rather than approximating.

In [ ]:
rc, _ = verith(
    "tests/limits/mods/m_countdown.py",
    "--buchi", "(= s0 0)",
    "--infer", "nuterm",             # no key, no network, no LLM
    "--pre-check", "cvc5",           # ... and cvc5 confirms it independently
)

$ uv run verith tests/limits/mods/m_countdown.py --buchi '(= s0 0)' --infer nuterm --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



[nuterm] columns: s0
[nuterm] training a ranking function that drops on every round the property fails
[nuterm] 991 sampled rounds, loss 0 -- certified
[nuterm] invariant: 2 of 3 conjuncts after pruning
[nuterm] inv: (and (>= s0 0) (<= s0 100))
[nuterm] ranking: (+ (* 2 (ite (> (+ 2 s0) 0) (+ 2 s0) 0)) (* 2 (ite (> (* 2 s0) 0) (* 2 s0) 0)) (* 2 (ite (> (* (- 2) s0) 0) (* (- 2) s0) 0)) (* 2 (ite (> (+ (- 1) (- s0)) 0) (+ (- 1) (- s0)) 0)) (* 2 (ite (> (- s0) 0) (- s0) 0)) (* 2 (ite (> (+ (- 1) s0) 0) (+ (- 1) s0) 0)) (* 2 (ite (> (- 1) 0) (- 1) 0)))
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         1 ms
   hrank     holds         3 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[1.5s, exit 0]


`[nuterm] inv:` and `[nuterm] ranking:` are the SMT-LIB the certificate is
written from — the invariant is the familiar `0 ≤ s0 ≤ 100`, and the ranking
function is the learned network written out, one `ite` per ReLU unit. It is
longer than the `s0` a human would write, and the three obligations underneath
it hold all the same.

The same Lean check as everywhere else in this notebook:

In [ ]:
if rc == 0:
    lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


### Neither route dominates

`m_step2` is the counterexample to nuterm being enough. It steps by two and
resets:

```c
x = 0;
while (true) { x = (x == 10) ? 0 : x + 2; }
```

`x == 0` recurs, so `G (F (= s0 0))` holds. But the invariant that makes it
provable is that `x` is **even**, and Houdini's lattice has no way to say so:
it offers signs, pairwise relations and the program's own constants, and a
congruence is outside all three. Without it, `x = 9` is a state the invariant
admits, it steps to `11`, and no rank drops on that.

In [ ]:
source("tests/limits/mods/m_step2.py", start="def init")
rc_nuterm, _ = verith(
    "tests/limits/mods/m_step2.py",
    "--buchi", "(= s0 0)",
    "--infer", "nuterm",
)

── tests/limits/mods/m_step2.py ──────────────────────────────────
def init():
    return 0


def update(old_x):
    if old_x == 10:
        return 0
    return old_x + 2


def module() -> Module:
    s = Var(Int([1, 1]))
    return Module.sequential(
        [s],
        convert_method(init, {}, [X(s)], theory=LIA),
        convert_method(update, {"old_x": s}, [X(s)], theory=LIA),
    )

$ uv run verith tests/limits/mods/m_step2.py --buchi '(= s0 0)' --infer nuterm -o /tmp/verith-tutorial.noindex -p Rea



[nuterm] columns: s0
[nuterm] training a ranking function that drops on every round the property fails
[nuterm] 833 sampled rounds, loss 0.9688 -- trained but not verified
[nuterm] training a ranking function that drops until the property is reached
[nuterm] 667 sampled rounds, loss 0 -- trained but not verified
error: --infer: no ranking function was certified. The rank is a non-negative sum of ReLUs, so it is convex in the state: a program whose run wraps around needs one that falls along the ramp and again across the reset, which no convex rank does. It is learned from rollouts too, so a property that never fails on one leaves nothing to train on.

[7.3s, exit 1]


Two attempts and a refusal naming the reason, in seconds. (The second attempt is
the route asking a weaker question — rank only the rounds *before* the property
is reached, and zero the rank where it holds — which is what makes a plain
wrap-around counter work. It does not rescue this one.)

Now the same module through the LLM loop. It costs an API round trip per
attempt, and cvc5 rejects a wrong proposal and hands the counterexample back:

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    rc_ai, _ = verith(
        "tests/limits/mods/m_step2.py",
        "--buchi", "(= s0 0)",
        "--infer", "ai-cegar",
        "--pre-check", "cvc5",
        timeout=600,
    )
else:
    rc_ai = None
    print("no ANTHROPIC_API_KEY -- skipping (see the setup cell)")

$ uv run verith tests/limits/mods/m_step2.py --buchi '(= s0 0)' --infer ai-cegar --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea



[CEGAR] attempt 0
  parse error: Could not parse RANKING `(ite (= s0 0) 0 (/ (- 12 s0) 2))`: Branches of the ITE must have comparable type.
[CEGAR] attempt 1
  inv: (and (>= s0 0) (<= s0 10) (= (mod s0 2) 0))
  ranking: (ite (= s0 0) 0 (- 12 s0))
[CEGAR] all obligations UNSAT — accepted
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         2 ms
   hrank     holds         2 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[17.0s, exit 0]


`(= (mod s0 2) 0)` — the model proposes the congruence directly, and cvc5
confirms it. The loop is doing real work here: attempt 0 was not even
well-sorted (`(/ (- 12 s0) 2)` puts an integer division against an integer
branch and cvc5 refuses the `ite`), and attempt 1 landed it. The route is
non-deterministic, so re-running this cell will not reproduce those two
attempts exactly — what is stable is that it gets there, and that every attempt
costs a round trip.

That is the trade, as the cells above measured it on one machine:

| | `m_countdown` | `m_step2` |
|---|---|---|
| `--infer nuterm` | **proved, 1.5 s** | refused, 7.3 s |
| `--infer ai-cegar` | proved, 5.7 s | **proved, 17.0 s** |

The gap widens with the size of the problem, because the learner's cost is
local — a rollout, a few hundred training epochs, an LP per region — while the
model's is a round trip per attempt, and it may need several. The learner also
answers offline and deterministically, which is what lets it run over a whole
corpus: it proves 45 of the 57 SV-COMP termination benchmarks, and 29 of the 32
safety properties the ic3ia route is meant to certify.

What it cannot do is invent a fact outside its candidate lattice. The model has
no lattice, so a congruence, a disjunction or a case split costs it nothing to
propose — and nothing but time when it guesses wrong, because cvc5 checks every
proposal before it reaches Lean.

So the order that makes sense is the order of cost: `--infer nuterm` first, and
the LLM for what it leaves behind. Section 8 adds two routes that change that
order again, because they can *prove* that a shape is empty — and hand the
proof to the LLM.

## 8. Searching a shape instead of proposing one (`--infer smt-linear`, `--infer sygus`)

Sections 6 and 7 both *propose*: the model proposes a certificate and cvc5
checks it, or the learner proposes a rank and a decision procedure certifies
it. Two more routes do neither. They fix the **shape** the certificate may
have and hand the whole space to cvc5 at once:

* **`--infer smt-linear`** fixes the arithmetic and leaves the coefficients
  open, so the search is one query — `exists c. forall s. obligations(c · s)`.
  `--buchi` asks it for a ranking function `c0 + c1*s0 + …` over a fixed
  invariant; `--safety` asks it for the invariant itself, as a conjunction of
  `a0 + a1*s0 + … >= 0` rows.
* **`--infer sygus`** (`--safety` only) fixes a **grammar** instead, and hands
  cvc5's SyGuS invariant track the three formulas `G P` is made of:
  `pre → inv`, `inv ∧ trans → inv'`, `inv → post`.

Neither needs a key, a network or a learner, and both are deterministic. But
the reason to reach for them is not that they are cheap — it is what they can
say when they **fail**, which is 8a — and what the next route does with
that, which is 8b.

In [ ]:
# The two routes below say more than the default `SAYS` greps for.
SHAPE = SAYS + ("[sygus]", "[smt-linear]", "resuming", "wrote artifact")

rc, _ = verith(
    "tests/limits/mods/m_countdown.py",
    "--safety", "(<= s0 100)",
    "--infer", "smt-linear",
    "--artifacts", "reset",          # start this section from an empty workspace
    "--pre-check", "cvc5",           # ... and cvc5 confirms it independently
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_countdown.py --safety '(<= s0 100)' --infer smt-linear --artifacts reset --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
[smt-linear] columns: s0
[smt-linear] inv: (>= (+ 100 (- s0)) 0)
   wrote artifact inv-0002-smt-linear.smt2 (proved)
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   wrote artifact system.smt2 (encoded)
   wrote artifact obligation-init_inv.smt2 (encoded)
   wrote artifact obligation-step_inv.smt2 (encoded)
   wrote artifact obligation-inv_imp_P.smt2 (encoded)
   init_inv  holds         1 ms
   step_inv  holds         1 ms
   inv_imp_P holds         1 ms
   wrote artifact inv.smt (encoded)
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.7s, exit 0]


One query, and the answer is the coefficients of a single row. `100 - s0 ≥ 0`
is the invariant a person would write, and it is written that way rather than
as the `200 - 2*s0 ≥ 0` the solver first handed over: the row is divided
through by its gcd, which is the same predicate and a smaller one to restate
in Lean.

The widths are tried one at a time — one row, then two, up to `--linear-rows`
— because the width is where the cost is. One row here is a few milliseconds
of solving; the 0.7 s the cell reports is `uv`, the module and the five
encodings around it. Asking for two rows at once does not finish in thirty
seconds, because every extra row multiplies a nonlinear search. So the first
width that answers wins — and each `unsat` on the way is kept, which is 8a.

The same Lean check as everywhere else in this notebook:


In [ ]:
if rc == 0:
    lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea

Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


### 8a. The answer that is worth more than a certificate

Ask for something that is not there. `m_toward5` walks toward 5 from both
sides, so a rank has to *branch* — and no linear function branches:

In [ ]:
rc_lin, _ = verith(
    "tests/limits/mods/m_toward5.py",
    "--buchi", "(= s0 5)",
    "--invariant", "(and (>= s0 0) (<= s0 10))",
    "--infer", "smt-linear",
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_toward5.py --buchi '(= s0 5)' --invariant '(and (>= s0 0) (<= s0 10))' --infer smt-linear -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
   wrote artifact inv.smt (encoded)
[smt-linear] columns: s0
   wrote artifact note-0010-smt-linear.md (no_solution)
error: --infer: --infer smt-linear found no ranking function. No ranking function linear in the state -- `c0 + c1*s0`, integer coefficients, no branching -- satisfies the two ranking obligations under the invariant `(and (>= s0 0) (<= s0 10))`. This is a proof that the space is empty, not a search that ran out of time: the query is `exists c. forall s. obligations(c)` over the coefficients, and cvc5 came back unsat. A ranking function for this module needs something outside that shape: a branch (`ite`), which is what a program whose run wraps around needs, or a stronger invariant to rank over. `--infer ai-cegar` and `--infer nuterm` both search shapes that have one.



That is not "the search gave up". `exists c. forall s. …` came back **unsat**,
which is a proof that *nothing* of that shape satisfies the obligations — a
fact about the module, established by a decision procedure, in a couple of
milliseconds. An LLM asked the same question would spend a round trip per
attempt discovering it, and would not be able to prove it at the end.

So it is written down. `artifacts/` is the project's workspace across runs
(`--artifacts use|ignore|reset`), and a refuted shape goes in it as a note
whose status is `no_solution` — kept apart from `unknown`, which is what a
search that merely ran out of budget leaves:

In [ ]:
notes = sorted((PROJECT / "artifacts").glob("note-*.md"))
print(notes[-1].name, "\n")
print(notes[-1].read_text())

note-0010-smt-linear.md 

No ranking function linear in the state -- `c0 + c1*s0`, integer coefficients, no branching -- satisfies the two ranking obligations under the invariant `(and (>= s0 0) (<= s0 10))`. This is a proof that the space is empty, not a search that ran out of time: the query is `exists c. forall s. obligations(c)` over the coefficients, and cvc5 came back unsat. A ranking function for this module needs something outside that shape: a branch (`ite`), which is what a program whose run wraps around needs, or a stronger invariant to rank over. `--infer ai-cegar` and `--infer nuterm` both search shapes that have one.



### 8b. Cascading: what one route rules out, the next one takes up

Here is the pattern the workspace is for. `m_step2` steps by two and resets,
so `x` is even in every reachable state — and section 7 showed `--infer
nuterm` refusing it, because a lattice of signs and pairwise relations cannot
say "even".

Neither can a conjunction of linear inequalities. `--infer smt-linear` does
not merely fail to find one; it **proves** there is none, at every width up to
`--linear-rows`, in milliseconds:

In [ ]:
rc_step2_lin, _ = verith(
    "tests/limits/mods/m_step2.py",
    "--safety", "(not (= s0 1))",
    "--infer", "smt-linear",
    "--artifacts", "reset",          # a clean workspace for the cascade
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_step2.py --safety '(not (= s0 1))' --infer smt-linear --artifacts reset -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
[smt-linear] columns: s0
[smt-linear] no invariant with 1 row(s); widening
[smt-linear] no invariant with 2 row(s); widening
   wrote artifact note-0002-smt-linear.md (no_solution)
error: --infer: --infer smt-linear found no invariant. No inductive invariant implying the property is a conjunction of 2 linear inequalities `a0 + a1*s0 >= 0`. This is a proof that the space is empty, not a search that ran out of time: the query is `exists c. forall s. obligations(c)` over the coefficients, and cvc5 came back unsat. Wider conjunctions were not tried: `--linear-rows` stopped at 2. An invariant may exist with more rows, or outside linear arithmetic altogether -- `--infer sygus` adds congruences (`x` even), which no conjunction of inequalities can state.

[0.6s, exit 1]


Now the second route, on the same module and the same property, into the same
project. Its grammar carries congruences — `(= (mod a0 + a1*s0 + … k) 0)`, for
the `k` the program itself mentions — which is exactly the atom the first
route proved it did not have:

In [ ]:
rc_step2_sygus, _ = verith(
    "tests/limits/mods/m_step2.py",
    "--safety", "(not (= s0 1))",
    "--infer", "sygus",
    "--pre-check", "cvc5",
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_step2.py --safety '(not (= s0 1))' --infer sygus --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea

[sygus] columns: s0
[sygus] grammar: congruence, constants [-11, -10, -9, -3, -2, -1, 0, 1, 2, 3, 9, 10, 11]
   wrote artifact inv-0003-sygus.smt2 (proved)
[sygus] inv: (= (mod (+ 2 (* 11 s0)) 2) 0)
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   wrote artifact system.smt2 (encoded)
   wrote artifact obligation-init_inv.smt2 (encoded)
   wrote artifact obligation-step_inv.smt2 (encoded)
   wrote artifact obligation-inv_imp_P.smt2 (encoded)
   init_inv  holds         1 ms
   step_inv  holds         2 ms
   inv_imp_P holds         1 ms
   wrote artifact inv.smt (encoded)
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.7s, exit 0]


In [ ]:
if rc_step2_sygus == 0:
    lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea

Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


That is the cascade: the cheap route could not close the obligations, said so
in a form the next run can read, and the route with the missing atom closed
them. Both halves are in the workspace — what was ruled out, and what was
found:

In [ ]:
import json as _json

index = _json.loads((PROJECT / "artifacts" / "index.json").read_text())
for e in sorted(index, key=lambda e: e["seq"]):
    if e["role"] in ("inv", "ranking", "note"):
        print(f"{e['role']:8s} {e['status']:12s} {e['producer']:11s} {e['name']}")

note     no_solution  smt-linear  note-0002-smt-linear.md
inv      proved       sygus       inv-0003-sygus.smt2
inv      encoded      sygus       inv.smt


The `inv` row is not only a record. It is written in SMT-LIB with the status a
later run filters on, so the next `verith` in this project **takes it as
given** — and for a safety property that is the whole certificate, so the run
below closes it without an LLM call even though `ai-cegar` is an LLM route:

In [ ]:
rc_resume, _ = verith(
    "tests/limits/mods/m_step2.py",
    "--safety", "(not (= s0 1))",
    "--infer", "ai-cegar",
    "--pre-check", "cvc5",
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_step2.py --safety '(not (= s0 1))' --infer ai-cegar --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea

.. resuming from inv-0003-sygus.smt2 (proved): taking it as the certificate
.. resuming from note-0002-smt-linear.md: a space an earlier run ruled out
[CEGAR] attempt 0
  inv: (= (mod (+ 2 (* 11 s0)) 2) 0)
[CEGAR] all obligations UNSAT — accepted
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   init_inv  holds         1 ms
   step_inv  holds         2 ms
   inv_imp_P holds         1 ms
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.9s, exit 0]


`.. resuming from inv-…-sygus.smt2 (proved)` is the handoff, and
`[CEGAR] all obligations UNSAT — accepted` is the loop finding it has nothing
left to ask. The store drops anything found for a different module, a
different property or a different proof rule before offering it, which is what
keeps a workspace from turning two identical command lines into two different
runs.

### 8c. The other direction: a refutation in an LLM's prompt

The same note that made 8b readable by a human is written to be read by a
model. `--infer ai-cegar` resumes `no_solution` notes and states them in its
prompt as established facts — so the attempt that would have gone on a linear
ranking function goes somewhere else instead. This needs a key, so it is
skipped when there is none:

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    verith(
        "tests/limits/mods/m_toward5.py",
        "--buchi", "(= s0 5)",
        "--invariant", "(and (>= s0 0) (<= s0 10))",
        "--infer", "smt-linear",
        "--artifacts", "reset",
        keep=SHAPE, limit=6,
    )
    print("\n" + "─" * 70 + "\n")
    rc_cascade, _ = verith(
        "tests/limits/mods/m_toward5.py",
        "--buchi", "(= s0 5)",
        "--invariant", "(and (>= s0 0) (<= s0 10))",
        "--infer", "ai-cegar",
        "--pre-check", "cvc5",
        keep=SHAPE, timeout=600,
    )
else:
    rc_cascade = None
    print("no ANTHROPIC_API_KEY — skipping (see the setup cell)")

$ uv run verith tests/limits/mods/m_toward5.py --buchi '(= s0 5)' --invariant '(and (>= s0 0) (<= s0 10))' --infer smt-linear --artifacts reset -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
   wrote artifact inv.smt (encoded)
[smt-linear] columns: s0
   wrote artifact note-0003-smt-linear.md (no_solution)
error: --infer: --infer smt-linear found no ranking function. No ranking function linear in the state -- `c0 + c1*s0`, integer coefficients, no branching -- satisfies the two ranking obligations under the invariant `(and (>= s0 0) (<= s0 10))`. This is a proof that the space is empty, not a search that ran out of time: the query is `exists c. forall s. obligations(c)` over the coefficients, and cvc5 came back unsat. A ranking function for this module needs something outside that shape: a branch (`ite`), which is what a program whose run wraps around needs, or a stronger invariant to rank over. `--infer ai-cegar` and `--infer nuterm` both search shape

Attempt **0**, and the model proposes
`(ite (= s0 5) 0 (ite (< s0 5) (- 5 s0) (- s0 5)))` — a rank that branches, on
a module where the note in front of it had just proved that nothing without a
branch can work. Compare section 7's `m_step2`, where the first attempt was
not even well-sorted: the LLM route is non-deterministic and this cell will
not reproduce those exact terms, but what it is being asked has changed. It is
no longer "find a ranking function"; it is "find one that is not in this
space", with the space named and the proof cited.

The two runs cost 0.6 s and 4.2 s here, and the 0.6 s is what stops the 4.2 s
from being spent twice.

### 8d. A state that is not integers

`--infer nuterm` reads scalar integers and refuses everything else by name.
The template route weighs **every** scalar component, whatever its sort: an
`Int` is itself, a `Bool` is `0`/`1` (`(ite s0 1 0)`, which is how a row says
`b` or `¬b`), and a bitvector is its unsigned value (`(ubv_to_int s0)`). One
integer template then covers a mixed state instead of one template per sort.

`m_boolint` is a Bool beside an Int — the flag chooses whether the counter
walks up or down:

In [ ]:
rc_bool, _ = verith(
    "tests/limits/mods/m_boolint.py",
    "--safety", "(and (>= s1 0) (<= s1 5))",
    "--infer", "smt-linear",
    "--artifacts", "reset",
    "--pre-check", "cvc5",
    keep=SHAPE,
)

$ uv run verith tests/limits/mods/m_boolint.py --safety '(and (>= s1 0) (<= s1 5))' --infer smt-linear --artifacts reset --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
[smt-linear] columns: (ite s0 1 0), s1
[smt-linear] no invariant with 1 row(s); widening
[smt-linear] inv: (and (>= (+ (ite s0 1 0) (* 3 s1)) 0) (>= (+ 5 (- s1)) 0))
   wrote artifact inv-0002-smt-linear.smt2 (proved)
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   wrote artifact system.smt2 (encoded)
   wrote artifact obligation-init_inv.smt2 (encoded)
   wrote artifact obligation-step_inv.smt2 (encoded)
   wrote artifact obligation-inv_imp_P.smt2 (encoded)
   init_inv  holds         1 ms
   step_inv  holds         2 ms
   inv_imp_P holds         2 ms
   wrote artifact inv.smt (encoded)
Project ready at: /tmp/verith-tutorial.noindex/Rea

[4.1s, exit 0]


A bitvector costs one thing more. Put one in the formula and cvc5 answers the
quantified query `unknown (INCOMPLETE)` in about a millisecond, at every
width — not a timeout, a refusal to state it. So the route asks the same
question a second way, as a **counterexample-guided loop** whose two halves
are both quantifier-free: propose coefficients that fit the states seen so
far, verify the proposal against the whole transition, and make the
counterexample the next sample.

An 8-bit counter to write it on — the module goes in the notebook's own
output directory rather than in the test corpus:

In [ ]:
BVMOD = OUT / "m_bvcount.py"
BVMOD.write_text('''# BV 8-bit counter: 0, 1, ... 10, then back to 0.
import torch
from zrth import Module, Term, Wire, BitVec, BV, Var, X


def module() -> Module:
    s = Var(BitVec(8, [1, 1]))
    one, ten, zero = (Wire(BitVec(8, [1, 1])) for _ in range(3))
    at_ten = Wire(BitVec(1, [1, 1]))      # BV has no Bool wires: one bit
    bumped = Wire(BitVec(8, [1, 1]))
    return Module.sequential(
        [s],
        [Term(BV.Const(torch.tensor([[0]])), [X(s)])],
        [Term(BV.Const(torch.tensor([[1]])), [one]),
         Term(BV.Const(torch.tensor([[10]])), [ten]),
         Term(BV.Const(torch.tensor([[0]])), [zero]),
         Term(BV.Eq(), [at_ten], [s, ten]),
         Term(BV.Add(), [bumped], [s, one]),
         Term(BV.Ite(), [X(s)], [at_ten, zero, bumped])],
    )
''')

rc_bv, _ = verith(
    str(BVMOD),
    "--safety", "(<= (ubv_to_int s0) 10)",
    "--infer", "smt-linear",
    "--artifacts", "reset",
    "--pre-check", "cvc5",
    keep=SHAPE,
)

$ uv run verith /tmp/verith-tutorial.noindex/m_bvcount.py --safety '(<= (ubv_to_int s0) 10)' --infer smt-linear --artifacts reset --pre-check cvc5 -o /tmp/verith-tutorial.noindex -p Rea

   wrote artifact property.smt (encoded)
[smt-linear] columns: (ubv_to_int s0)
[smt-linear] cvc5 will not state the quantified query for this module; searching coefficients in [-24, 24] by counterexample instead
[smt-linear] inv: (>= (+ 10 (- (ubv_to_int s0))) 0)
   wrote artifact inv-0002-smt-linear.smt2 (proved)
.. SMT pre-check (cvc5): <=5000 ms per query, <=20000 ms total
   wrote artifact system.smt2 (encoded)
   wrote artifact obligation-init_inv.smt2 (encoded)
   wrote artifact obligation-step_inv.smt2 (encoded)
   wrote artifact obligation-inv_imp_P.smt2 (encoded)
   init_inv  holds         1 ms
   step_inv  holds         2 ms
   inv_imp_P holds         1 ms
   wrote artifact inv.smt (encoded)
Project ready at: /tmp/verith-tutorial.noindex/Rea

[0.7s, exit 0]


In [ ]:
if rc_bv == 0:
    lake_build()

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea

Build completed successfully (3466 jobs).


=> PROVED (lake exit 0)


The loop needs a *finite* candidate space to terminate, so the coefficients
are bounded — read off the constants the program mentions — and a refutation
from it is a proof about that box rather than about every integer. The note
says which, because the difference is exactly what a later prompt must not be
told wrongly.

Where the four no-LLM routes sit, then:

| | finds | answers *no* | state it reads |
|---|---|---|---|
| `--infer nuterm` | Houdini's lattice + a learned rank | no — it refuses, without deciding the space | scalar `Int` |
| `--infer smt-linear` | coefficients of a fixed linear shape | **yes, as a proof** (bounded, when the loop answered) | scalar `Int`, `Bool`, `BitVec` |
| `--infer sygus` | anything its grammar generates, congruences included | **yes, as a proof** — the grammar is finite | scalar `Int` |
| `--infer fbk-proveit` | ic3ia's interpolants | no | whatever the NA encoding expresses |

The order that pays is still the order of cost — the cheap decisive routes
first, the LLM for what they leave behind — with one addition: run the routes
that can say *no* first, because what they rule out is what the expensive
route no longer has to try.

## 9. Letting ic3ia find the invariant (`--fbk-proveit`)

For a safety property there is a third option, with no LLM in it at all: hand
the module to a model checker. `--fbk-proveit` certifies through the
`proveit.py` driver of `lean-ltl-certifying`:

```
module.py ──verith──▶ <Proj>/ProveIt/ReaNA.lean      (an NA-shaped encoding)
                            │
                            ▼   proveit.py
                   lean2vmt ──▶ ic3ia ──▶ vmt2lean
                            │
                            ▼
                   <Proj>/Certificate/Certificate.lean
```

ic3ia decides reachability, so this route is `--safety` only — `--buchi` is
rejected — and it supplies the invariant itself, so `--invariant`, `--ranking`,
`--pre` and `--infer` are rejected too. It needs MathSAT's Python bindings, an
ic3ia build and a `lean-ltl-certifying` checkout on the same Lean toolchain;
all three are configured in the setup cell.

`verith` also emits `Certificate/Equivalence.lean`, the proof that the NA model
ic3ia reasoned about really is this module. Without it, a mistranslation
anywhere along the SMT path would leave a machine-checked certificate about a
*different* system.

In [ ]:
FBK = IC3IA.exists() and PROVEIT.is_dir() and MATHSAT.is_dir()
if FBK:
    rc, _ = verith(
        "tests/limits/mods/m_countdown.py",
        "--safety", "(and (>= s0 0) (<= s0 100))",   # no invariant: ic3ia finds it
        "--fbk-proveit", PROVEIT,
        "--ic3ia", IC3IA,
    )
    FBK = rc == 0
    if FBK:
        print("\nCertificate/ now holds:",
              sorted(p.name for p in (PROJECT / "Certificate").iterdir()))
else:
    print("ic3ia / lean-ltl-certifying / mathsat not configured -- skipping (see the setup cell)")

$ uv run verith tests/limits/mods/m_countdown.py --safety '(and (>= s0 0) (<= s0 100))' --fbk-proveit /Users/marek/zeroth/proof-prototyping/lean-ltl-certifying --ic3ia /Users/marek/zeroth/fbk/ic3ia/build/ic3ia -o /tmp/verith-tutorial.noindex -p Rea



    ✔ property is SAFE
    ✔ invariant witness: /var/folders/p2/6pl6scg57lb35sp3r_ncs69h0000gn/T/certify-ReaNA-zr7sih37/ReaNA_witness.smt2
    ✔ certificate: /tmp/verith-tutorial.noindex/Rea/ProveIt/ReaCert.lean (3936 bytes)
Installed certificate: /tmp/verith-tutorial.noindex/Rea/Certificate/Certificate.lean
Wrote the model-is-the-module proof: /tmp/verith-tutorial.noindex/Rea/Certificate/Equivalence.lean
Project ready at: /tmp/verith-tutorial.noindex/Rea

[5.8s, exit 0]

Certificate/ now holds: ['Certificate.lean', 'Equivalence.lean']


This project's lakefile requires the `lean-ltl-certifying` checkout, which the
warm manifest has never heard of, so the build has to resolve it with
`lake update` — the only `lake` command in this notebook that goes to the
network. The resolved manifest is cached, so a re-run does not.

In [ ]:
FBK_MANIFEST = OUT / "lake-manifest-fbk.json"
if FBK:
    if FBK_MANIFEST.exists():
        rc, _ = lake_build(manifest=FBK_MANIFEST)
    else:
        rc, _ = lake_build(manifest=None)          # `lake update`, then build
        if rc == 0:
            shutil.copy2(PROJECT / "lake-manifest.json", FBK_MANIFEST)

$ lake build Certificate      # in /tmp/verith-tutorial.noindex/Rea



Build completed successfully (3473 jobs).


=> PROVED (lake exit 0)


## 10. Flag reference

| flag | default | meaning |
|---|---|---|
| `-o` / `--output-dir` | `.` | where the project is created |
| `-p` / `--project-name` | `Rea` | Lean package name; the project is `<-o>/<-p>` |
| `-d` / `--module-def` | `module` | name of the callable in the module file |
| `-x` / `--executable` | off | also emit `Main.lean` and a `lean_exe` target |
| `--safety FORMULA` | — | `G FORMULA`, proved by an invariant alone |
| `--buchi FORMULA` | — | `G (F FORMULA)`, proved by invariant + ranking |
| `--invariant` | — | SMT-LIB 2 Bool over `s0…` |
| `--ranking` | — | SMT-LIB 2 Int over `s0…`; `--buchi` only |
| `--pre` | — | SMT-LIB 2 Bool over `e0…` / `el0…`, added to `init_pre` and `update_pre` |
| `--pre-check cvc5` | `none` | ask cvc5 whether the obligations hold, before generating |
| `--smt-tactics cvc5` | `none` | let cvc5 settle branch conditions and hint `nlinarith` |
| `--smt-timeout`, `--smt-budget` | 5000, 20000 ms | per-query and per-phase limits for both of the above |
| `--infer [ai\|ai-cegar\|nuterm\|sygus\|smt-linear\|fbk-proveit]` | `ai-cegar` | which route finds the certificate: an LLM, (`nuterm`) a learned and certified ranking function, (`sygus`) an invariant synthesised over a grammar, (`smt-linear`) the coefficients of a fixed linear shape, or (`fbk-proveit`) ic3ia |
| `--sygus-grammar`, `--sygus-conjuncts` | `congruence`, 3 | `--infer sygus`: what an atom may be, and how many of them — the bound is what makes the space finite, and so decidably empty |
| `--linear-rows N` | 2 | `--infer smt-linear --safety`: how many linear inequalities the invariant may conjoin; tried one width at a time |
| `--model`, `--base-url` | `claude-sonnet-4-6`, — | which LLM, and which endpoint; rejected by a route that calls none |
| `--build-cert` | off | `lake update` + `lake build Certificate`, failing on a surviving `sorry` |
| `--proveit-dir DIR` | — | `--infer fbk-proveit`: the `lean-ltl-certifying` checkout; `--safety` only. `--fbk-proveit DIR` is the older spelling of both flags at once |
| `--ic3ia PATH` | `$IC3IA`, then `PATH` | the ic3ia executable |
| `--fbk-simplify`, `--fbk-equiv` | `cvc5`, `lean` | run cvc5's rewriter on the NA model; emit the model-is-the-module proof |
| `--artifacts` | `use` | the project's `artifacts/` across runs: `use` resumes from what an earlier run left, `ignore` searches afresh, `reset` empties it |
| `--cert-file PATH` | — | a standalone certificate `.lean` instead of a project |
| `--hammer-file PATH` | — | write `ZerothHammer.lean` alone and exit |

`uv run verith --help` prints the same list with the full descriptions and a
set of copy-pasteable examples.

## Where to go next

* [`verith_gym.md`](verith_gym.md) — the same CLI from the gymnasium side:
  writing an environment that extracts into integer arithmetic, matrix-shaped
  state, and more on `--infer` backends.
* [`../python/tests/BENCHMARKS.md`](../python/tests/BENCHMARKS.md) — every
  `verith` invocation the two measurement harnesses make, one per case,
  copy-pasteable.
* [`../python/tests/limits/README.md`](../python/tests/limits/README.md) — what
  currently verifies and what does not, and the shared-`.lake` harness this
  notebook borrows its setup from, scaled to 77 cases.
* [`../python/tests/lean/fbk/README.md`](../python/tests/lean/fbk/README.md) —
  building MathSAT, ic3ia and `lean-ltl-certifying` for section 9.